In [ ]:
# 01 — Normalize coffee sales

Source: `01_original.csv`

| Feature | Rule |
|---|---|
| `money` | min-max to `[0, 1]` |
| `weekday` | Monday=0 … Sunday=6 |
| `is_weekend` | 1 if Saturday/Sunday, else 0 |
| `day_period` | early morning=0 (00–05), morning=1 (06–11), afternoon=2 (12–17), night=3 (18–23) |
| `hour` | 0–23 |
| `cash_type` | `card`=0, `cash`=1 |
| `coffee_name` | one-hot |
| `card` | dropped |

`datetime` was exported from Excel as `mm:ss.0`, which strips the hour (`15:50.5` is `10:15:50.5`). Hours are recovered from `ref_full_datetime.csv` (same 3636 rows, same order).

In [ ]:
from pathlib import Path

import pandas as pd

df = pd.read_csv("01_original.csv")

# --- date features ---
df["date"] = pd.to_datetime(df["date"], format="%d/%m/%Y")
df["weekday"] = df["date"].dt.dayofweek
df["is_weekend"] = df["weekday"].isin([5, 6]).astype(int)

# --- hour: recover from original timestamps (Excel stripped HH) ---
REF_LOCAL = Path("ref_full_datetime.csv")
REF_URL = (
    "https://raw.githubusercontent.com/mehmetkahya0/"
    "coffee-shop-business-intelligence/main/archive-datas/2024-to-2025sales.csv"
)
full = pd.read_csv(REF_LOCAL if REF_LOCAL.exists() else REF_URL)
assert len(full) == len(df), "Reference dataset size does not match 01_original.csv"
assert (pd.to_datetime(full["date"]).dt.normalize() == df["date"]).all()
assert (full["money"].astype(float).values == df["money"].astype(float).values).all()
assert (full["coffee_name"].values == df["coffee_name"].values).all()

df["hour"] = pd.to_datetime(full["datetime"]).dt.hour

# early_morning 00-05 | morning 06-11 | afternoon 12-17 | night 18-23
df["day_period"] = pd.cut(
    df["hour"],
    bins=[-1, 5, 11, 17, 23],
    labels=[0, 1, 2, 3],
).astype(int)

# --- cash_type ---
df["cash_type"] = df["cash_type"].map({"card": 0, "cash": 1}).astype(int)

# --- money: min-max ---
money_min, money_max = df["money"].min(), df["money"].max()
df["money"] = (df["money"] - money_min) / (money_max - money_min)

# --- coffee_name: one-hot ---
coffee = pd.get_dummies(df["coffee_name"], prefix="coffee_name", dtype=int)

normalized = pd.concat(
    [
        df[["money", "weekday", "is_weekend", "day_period", "hour", "cash_type"]],
        coffee,
    ],
    axis=1,
)

normalized.to_csv("02_normalized.csv", index=False)
normalized

,money,weekday,is_weekend,day_period,hour,cash_type,coffee_name_Americano,coffee_name_Americano with Milk,coffee_name_Cappuccino,coffee_name_Cocoa,coffee_name_Cortado,coffee_name_Espresso,coffee_name_Hot Chocolate,coffee_name_Latte
0,0.940585,4,0,1,10,0,0,0,0,0,0,0,0,1
1,0.940585,4,0,2,12,0,0,0,0,0,0,0,1,0
2,0.940585,4,0,2,12,0,0,0,0,0,0,0,1,0
3,0.492687,4,0,2,13,0,1,0,0,0,0,0,0,0
4,0.940585,4,0,2,13,0,0,0,0,0,0,0,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3631,0.806216,6,1,1,10,0,0,0,1,0,0,0,0,0
3632,0.806216,6,1,2,14,0,0,0,0,1,0,0,0,0
3633,0.806216,6,1,2,14,0,0,0,0,1,0,0,0,0
3634,0.358318,6,1,2,15,0,1,0,0,0,0,0,0,0
